# Setup

In [2]:
import sqlite3
import pandas as pd

conn = sqlite3.connect(":memory:")
cursor = conn.cursor()

# Customer Order Table

In [3]:
cursor.execute("""
CREATE TABLE customer_orders (
    order_id INTEGER,
    customer_id INTEGER,
    customer_name TEXT,
    city TEXT,
    revenue REAL
)
""")

# Insert Data

In [4]:
cursor.executemany("""
INSERT INTO customer_orders VALUES (?, ?, ?, ?, ?)
""", [
    (1, 101, 'Amit', 'Mumbai', 5000),
    (2, 102, 'Neha', 'Pune', 7000),
    (3, 101, 'Amit', 'Mumbai', 3000),
    (4, 103, 'Rahul', 'Delhi', 9000),
    (5, 102, 'Neha', 'Pune', 2000),
    (6, 104, 'Priya', 'Mumbai', 4000),
    (7, 105, 'Karan', 'Pune', 6000),
    (8, 103, 'Rahul', 'Delhi', 5000),
    (9, 101, 'Amit', 'Mumbai', 2000),
    (10, 103, 'Rahul', 'Delhi', 3000)
])

conn.commit()

## Project Overview

## Data Preview

In [5]:
pd.read_sql_query(
    "SELECT * FROM customer_orders",
    conn
)

,order_id,customer_id,customer_name,city,revenue
0,1,101,Amit,Mumbai,5000.0
1,2,102,Neha,Pune,7000.0
2,3,101,Amit,Mumbai,3000.0
3,4,103,Rahul,Delhi,9000.0
4,5,102,Neha,Pune,2000.0
5,6,104,Priya,Mumbai,4000.0
6,7,105,Karan,Pune,6000.0
7,8,103,Rahul,Delhi,5000.0
8,9,101,Amit,Mumbai,2000.0
9,10,103,Rahul,Delhi,3000.0


## KPI 1 - Customer Lifetime Value (CLV)

In [6]:
query = """
SELECT customer_id,
       customer_name,
       SUM(revenue) AS lifetime_value
FROM customer_orders
GROUP BY customer_id, customer_name
ORDER BY lifetime_value DESC
"""

pd.read_sql_query(query, conn)

,customer_id,customer_name,lifetime_value
0,103,Rahul,17000.0
1,101,Amit,10000.0
2,102,Neha,9000.0
3,105,Karan,6000.0
4,104,Priya,4000.0


## KPI 2 - Repeat Customers

In [7]:
query = """
SELECT customer_id,
       customer_name,
       COUNT(order_id) AS order_count
FROM customer_orders
GROUP BY customer_id, customer_name
HAVING COUNT(order_id) > 1
ORDER BY order_count DESC
"""

pd.read_sql_query(query, conn)

,customer_id,customer_name,order_count
0,101,Amit,3
1,103,Rahul,3
2,102,Neha,2


## KPI 3 - Revenue By City

In [8]:
query = """
SELECT city,
       SUM(revenue) AS total_revenue
FROM customer_orders
GROUP BY city
ORDER BY total_revenue DESC
"""

pd.read_sql_query(query, conn)

,city,total_revenue
0,Delhi,17000.0
1,Pune,15000.0
2,Mumbai,14000.0


Delhi generated the highest revenue (₹17,000), indicating strong customer value concentration in that city. Future marketing campaigns should prioritize Delhi to maximize ROI.

## KPI 4 - Customer Segmentation

In [9]:
query = """
SELECT
    customer_id,
    customer_name,
    SUM(revenue) AS lifetime_value,

    CASE
        WHEN SUM(revenue) >= 15000 THEN 'High Value'
        WHEN SUM(revenue) >= 8000 THEN 'Medium Value'
        ELSE 'Low Value'
    END AS customer_segment

FROM customer_orders

GROUP BY customer_id, customer_name

ORDER BY lifetime_value DESC
"""

pd.read_sql_query(query, conn)

,customer_id,customer_name,lifetime_value,customer_segment
0,103,Rahul,17000.0,High Value
1,101,Amit,10000.0,Medium Value
2,102,Neha,9000.0,Medium Value
3,105,Karan,6000.0,Low Value
4,104,Priya,4000.0,Low Value


Customer segmentation shows that Rahul is the only High Value customer. Amit and Neha are Medium Value customers with growth potential, while Karan and Priya are Low Value customers who may require targeted engagement campaigns.

## KPI 5 - Revenue Contribution

In [10]:
query = """
WITH customer_revenue AS (
    SELECT
        customer_id,
        customer_name,
        SUM(revenue) AS lifetime_value
    FROM customer_orders
    GROUP BY customer_id, customer_name
)

SELECT
    customer_id,
    customer_name,
    lifetime_value,

    ROUND(
        lifetime_value * 100.0 /
        (SELECT SUM(lifetime_value) FROM customer_revenue),
        2
    ) AS revenue_percentage

FROM customer_revenue

ORDER BY revenue_percentage DESC
"""

pd.read_sql_query(query, conn)

,customer_id,customer_name,lifetime_value,revenue_percentage
0,103,Rahul,17000.0,36.96
1,101,Amit,10000.0,21.74
2,102,Neha,9000.0,19.57
3,105,Karan,6000.0,13.04
4,104,Priya,4000.0,8.70



This customer analytics project examined customer revenue, repeat purchase behavior, customer segmentation, and revenue contribution.

Key Findings

- Top Customer: Rahul
- Lifetime Value: ₹17,000
- Revenue Contribution: 36.97%
- Top City: Delhi
- Repeat Customers: 3

Recommendations

- Retain Rahul through loyalty initiatives.
- Convert Medium Value customers into High Value customers.
- Focus customer acquisition efforts in Delhi.
- Increase repeat purchases from Low Value customers.

# Pandas Analysis

In [12]:
### load Customer REvnue data ###
customer_df = pd.read_sql_query("""
SELECT
    customer_id,
    customer_name,
    city,
    SUM(revenue) AS lifetime_value
FROM customer_orders
GROUP BY customer_id, customer_name, city
""", conn)

customer_df

,customer_id,customer_name,city,lifetime_value
0,101,Amit,Mumbai,10000.0
1,102,Neha,Pune,9000.0
2,103,Rahul,Delhi,17000.0
3,104,Priya,Mumbai,4000.0
4,105,Karan,Pune,6000.0


In [13]:
### sort Customers by value ###
customer_df.sort_values(
    by="lifetime_value",
    ascending=False
)

,customer_id,customer_name,city,lifetime_value
2,103,Rahul,Delhi,17000.0
0,101,Amit,Mumbai,10000.0
1,102,Neha,Pune,9000.0
4,105,Karan,Pune,6000.0
3,104,Priya,Mumbai,4000.0


Insight -
Rahul is the highhest value customer with 17000 Rupees lifetime revenue

In [14]:
### Segament Distribution ###
customer_df["segment"] = customer_df["lifetime_value"].apply(
    lambda x: "High Value" if x >= 15000
    else "Medium Value" if x >= 8000
    else "Low Value"
)

customer_df

,customer_id,customer_name,city,lifetime_value,segment
0,101,Amit,Mumbai,10000.0,Medium Value
1,102,Neha,Pune,9000.0,Medium Value
2,103,Rahul,Delhi,17000.0,High Value
3,104,Priya,Mumbai,4000.0,Low Value
4,105,Karan,Pune,6000.0,Low Value


In [15]:
### Count Segment ###
customer_df["segment"].value_counts()

segment
Medium Value    2
Low Value       2
High Value      1
Name: count, dtype: int64

Insight:

Most customers fall into the Medium and Low Value segments, indicating opportunities for customer growth initiatives.

In [16]:
### Revenue by City ###
city_df = customer_df.groupby(
    "city"
)["lifetime_value"].sum().reset_index()

city_df.sort_values(
    by="lifetime_value",
    ascending=False
)

,city,lifetime_value
0,Delhi,17000.0
2,Pune,15000.0
1,Mumbai,14000.0


Insight:

Delhi generates the highest customer revenue

In [17]:
### Revenue Contribution ###
customer_df["revenue_pct"] = round(
    customer_df["lifetime_value"] /
    customer_df["lifetime_value"].sum() * 100,
    2
)

customer_df.sort_values(
    by="revenue_pct",
    ascending=False
)

,customer_id,customer_name,city,lifetime_value,segment,revenue_pct
2,103,Rahul,Delhi,17000.0,High Value,36.96
0,101,Amit,Mumbai,10000.0,Medium Value,21.74
1,102,Neha,Pune,9000.0,Medium Value,19.57
4,105,Karan,Pune,6000.0,Low Value,13.04
3,104,Priya,Mumbai,4000.0,Low Value,8.70


Insight:

Rahul contributes nearly 37% of total revenue.

## Bussiness Insight 

1. Rahul is the highest-value customer with a lifetime value of ₹17,000.

2. Rahul contributes 36.97% of total company revenue.

3. Delhi is the highest-performing city, generating ₹17,000 in revenue.

4. Three customers are repeat customers, indicating healthy customer retention.

5. Only one customer falls into the High Value segment, suggesting an opportunity to grow Medium Value customers into High Value customers.

## Executive Summary
Executive Summary

This project analyzed customer behavior using SQL and Pandas.

Key Findings

- Top Customer: Rahul
- Lifetime Value: ₹17,000
- Revenue Contribution: 36.97%
- Top City: Delhi
- Repeat Customers: 3

Recommendations

- Retain high-value customers.
- Develop loyalty programs for repeat customers.
- Focus marketing efforts in Delhi.
- Convert Medium Value customers into High Value customers.